In [1]:
import numpy as np
import os
import pandas as pd
import scanpy as sc
from collections import Counter
from scipy import sparse
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `annda

In [2]:
# Main loop for enrichment analysis
samples = ["Xenium_5K_BC", "Xenium_5K_OC", "Xenium_5K_CC", "Xenium_5K_LC", "Xenium_5K_Prostate"]
pseudocount = 0.1
target_genes_dict = {}

for sample in samples:
    
    print("="* 20 + f"Processing sample: {sample}" + "="* 20)
    
    # Paths
    intermediate_path = f"../../data/{sample}/intermediate_data/"
    processed_path = f"../../data/{sample}/processed_data/"
    output_path = f"../../output/1_exploration/{sample}/"
    os.makedirs(output_path, exist_ok=True)

    # Read data
    adata = sc.read_h5ad(intermediate_path + "adata.h5ad")
    cell_ids = np.load(processed_path + "cell_ids.npy", allow_pickle=True)
    assert adata.shape[0] > cell_ids.shape[0], "cell_ids should be a subset of adata.obs_names"

    # Tumor cells
    adata_tumor = adata[adata.obs["cell_id"].isin(cell_ids)].copy()

    # Read nuclear and cytoplasmic expression matrices
    X_nuc = sparse.load_npz(processed_path + "nuclear_expression_matrix.npz")
    X_cyto = sparse.load_npz(processed_path + "cytoplasmic_expression_matrix.npz")

    assert adata_tumor.shape[0] == X_nuc.shape[0], "adata_tumor and X_nuc should have the same number of cells"
    assert X_nuc.shape[0] == X_cyto.shape[0], "Nuclear and cytoplasmic expression matrices should have the same number of cells"
    assert X_nuc.shape[1] == X_cyto.shape[1], "Nuclear and cytoplasmic expression matrices should have the same number of genes"
    
    # Compute cytoplasmic fraction
    nuc_counts = np.asarray(X_nuc.sum(axis=0)).ravel()
    cyto_counts = np.asarray(X_cyto.sum(axis=0)).ravel()
    cyto_fraction = cyto_counts / (cyto_counts + nuc_counts + 1e-12)

    # Compute log2 fold change between cytoplasmic and nuclear counts
    nuc_total = nuc_counts.sum()
    cyto_total = cyto_counts.sum()

    nuc_cpm = nuc_counts / nuc_total * 1e6
    cyto_cpm = cyto_counts / cyto_total * 1e6
    log2fc = np.log2((cyto_cpm + pseudocount) / (nuc_cpm + pseudocount))
    
    # Compute library sizes, normalize the expression matrices, log transform, and convert to CSC format for efficient column slicing
    nuc_libsize = np.asarray(X_nuc.sum(axis=1)).ravel()
    cyto_libsize = np.asarray(X_cyto.sum(axis=1)).ravel()

    nuc_norm = X_nuc.multiply(1e4 / np.maximum(nuc_libsize, 1)[:, None]).tocsr()
    cyto_norm = X_cyto.multiply(1e4 / np.maximum(cyto_libsize, 1)[:, None]).tocsr()

    nuc_log = nuc_norm.copy()
    cyto_log = cyto_norm.copy()

    nuc_log.data = np.log1p(nuc_log.data)
    cyto_log.data = np.log1p(cyto_log.data)
    
    nuc_log_csc = nuc_log.tocsc()
    cyto_log_csc = cyto_log.tocsc()

    # Perform Wilcoxon signed-rank test for each gene
    n_genes = nuc_log_csc.shape[1]
    stats = np.full(n_genes, np.nan)
    pvalues = np.full(n_genes, np.nan)

    for j in range(n_genes):
        x_nuc = nuc_log_csc[:, j].toarray().ravel()
        x_cyto = cyto_log_csc[:, j].toarray().ravel()
        diff = x_cyto - x_nuc
        if np.all(diff == 0):
            stats[j] = 0
            pvalues[j] = 1
        else:
            res = wilcoxon(x_cyto, x_nuc, alternative="two-sided")
            stats[j] = res.statistic
            pvalues[j] = res.pvalue
    fdr = multipletests(pvalues, method="fdr_bh")[1]
    
    # Create a DataFrame to store the results
    result = pd.DataFrame({
        "gene": adata_tumor.var_names,
        "nuclear_counts": nuc_counts,
        "cytoplasmic_counts": cyto_counts,
        "total_counts": nuc_counts + cyto_counts,
        "cytoplasmic_fraction": cyto_fraction,
        "nuclear_CPM": nuc_cpm,
        "cytoplasmic_CPM": cyto_cpm,
        "log2FC_cyto_vs_nuc": log2fc,
        "wilconxon_stat": stats,
        "pvalue": pvalues,
        "FDR": fdr
    })

    # Filter out genes with low total counts (bottom 10%)
    total_count_thr = np.percentile(result["total_counts"], 10)  # Filter out genes with low total counts (bottom 10%)
    print(f"Filtering out genes with total counts below {total_count_thr:.2f} (bottom 10%)")

    result = result[result["total_counts"] >= total_count_thr]
    result = result.sort_values("log2FC_cyto_vs_nuc", ascending=False)
    result.to_csv(output_path + "enrichment_results.csv", index=False)
    
    # Identify genes enriched in cytoplasm based on log2 fold change and FDRß
    criteria = (result["log2FC_cyto_vs_nuc"] > 0) & (result["FDR"] < 0.05)
    target_genes_dict[sample] = result[criteria]["gene"].tolist()
    print(f"Number of genes enriched in cytoplasm: {result[criteria].shape[0]}")

====================Processing sample: Xenium_5K_BC====================
Filtering out genes with total counts below 335.00 (bottom 10%)
Number of genes enriched in cytoplasm: 1158
====================Processing sample: Xenium_5K_OC====================
Filtering out genes with total counts below 536.00 (bottom 10%)
Number of genes enriched in cytoplasm: 1053
====================Processing sample: Xenium_5K_CC====================
Filtering out genes with total counts below 259.00 (bottom 10%)
Number of genes enriched in cytoplasm: 1244
====================Processing sample: Xenium_5K_LC====================
Filtering out genes with total counts below 206.00 (bottom 10%)
Number of genes enriched in cytoplasm: 1612
====================Processing sample: Xenium_5K_Prostate====================
Filtering out genes with total counts below 205.00 (bottom 10%)
Number of genes enriched in cytoplasm: 1207


In [3]:
# Overlap genes
overlap_genes = set(target_genes_dict[samples[0]])
for sample in samples[1:]:
    overlap_genes.intersection_update(set(target_genes_dict[sample]))

enrichment_path = "../../output/1_exploration/enrichment/"
os.makedirs(enrichment_path, exist_ok=True)
np.save(enrichment_path + "overlap_genes_all.npy", np.array(list(overlap_genes)))
print(f"Number of genes overlapping in all samples: {len(overlap_genes)}")

Number of genes overlapping in all samples: 503


In [4]:
# Overlap genes across at least three samples
gene_counts = Counter()
for sample in samples:
    gene_counts.update(set(target_genes_dict[sample]))
overlap_genes = {gene for gene, count in gene_counts.items() if count >= 3}

np.save(enrichment_path + "overlap_genes_3plus_samples.npy", np.array(list(overlap_genes)))
print(f"Number of genes overlapping in at least 3 samples: {len(overlap_genes)}")

Number of genes overlapping in at least 3 samples: 1129
